# 02 — Silver: frontalieri puliti (Bronze → Silver)

**Cosa fa:** legge gli ultimi file scaricati dal robot GitHub (cartella `bronze` del repository),
li pulisce, controlla la qualità e li salva come tabelle Delta in `workspace.frontalieri`.

**Prerequisito:** il repository `geopolitica-dati` è collegato a Databricks come *Git folder*
(e prima di eseguire: menu del Git folder → **Pull**, per avere gli ultimi dati).

**Output:** `silver_frontalieri_cantone` e `silver_frontalieri_settori`.


In [ ]:
# --- PARAMETRI ---------------------------------------------------------------
import os, glob, json
import pandas as pd

REPO_ROOT = "/Workspace/Users/luca.tarantino@hotmail.com/geopolitica-dati"
BRONZE_DIR = f"{REPO_ROOT}/bronze/bfs_sse/DF_GGS_1"
CATALOG, SCHEMA = "workspace", "frontalieri"

assert os.path.isdir(BRONZE_DIR), (
    f"Cartella non trovata: {BRONZE_DIR}\n"
    "Controlla che il Git folder si chiami 'geopolitica-dati' e sia nella tua cartella Home, poi fai Pull.")
print("OK, cartella Bronze trovata:")
for f in sorted(os.listdir(BRONZE_DIR)):
    print("  ", f)


In [ ]:
# --- 1. SCEGLI I FILE PIÙ RECENTI ---------------------------------------------
# I nomi iniziano con un timestamp (es. 20261009T181500Z_...), quindi l'ordine
# alfabetico coincide con l'ordine cronologico: l'ultimo è il più recente.
def ultimo(suffisso):
    files = sorted(glob.glob(f"{BRONZE_DIR}/*_{suffisso}"))
    assert files, f"Nessun file *_{suffisso} in {BRONZE_DIR}"
    return files[-1]

f_cantone = ultimo("totale_per_cantone.csv.gz")
f_settori = ultimo("settori_cantoni_confine.csv.gz")
f_prov = ultimo("provenance.json")
prov = json.load(open(f_prov))
print("Dati del download:", prov["scaricato_utc"])
print(" -", os.path.basename(f_cantone))
print(" -", os.path.basename(f_settori))


In [ ]:
# --- 2. PULIZIA ----------------------------------------------------------------
# Regole (documentate, nessuna correzione "creativa"):
#  - teniamo codici + etichette, scartiamo le colonne tecniche SDMX (STRUCTURE, ACTION, DECIMALS...)
#  - nomi colonna in italiano, minuscoli, senza spazi
#  - il valore diventa numero decimale; NON lo arrotondiamo (l'UST pubblica stime con decimali)
#  - dal trimestre ("2026-Q2") ricaviamo anno, numero di trimestre e data di fine trimestre
RINOMINA = {
    "NOGA": "noga_cod", "NOGA 2008": "noga_desc",
    "CNTRY": "paese_cod", "Paese di residenza": "paese_desc",
    "SEX": "sesso_cod",
    "WORK_CANTON": "cantone_cod", "Cantone svizzero": "cantone_desc",
    "TIME_PERIOD": "trimestre",
    "OBS_VALUE": "frontalieri",
    "OBS_STATUS": "stato_oss",
}

def pulisci(path):
    df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    mancanti = set(RINOMINA) - set(df.columns)
    assert not mancanti, f"Colonne attese mancanti (la fonte è cambiata?): {mancanti}"
    df = df[list(RINOMINA)].rename(columns=RINOMINA)
    df["frontalieri"] = pd.to_numeric(df["frontalieri"].replace("", None), errors="coerce")
    df["anno"] = df["trimestre"].str[:4].astype(int)
    df["trim"] = df["trimestre"].str[-1].astype(int)
    df["data_fine_trimestre"] = (pd.PeriodIndex(df["trimestre"].str.replace("-", ""), freq="Q")
                                 .end_time.normalize().date)
    df["fonte_file"] = os.path.basename(path)
    df["scaricato_utc"] = prov["scaricato_utc"]
    return df

cantone = pulisci(f_cantone)
settori = pulisci(f_settori)
print("cantone:", cantone.shape, "| settori:", settori.shape)
cantone.head()


In [ ]:
# --- 3. CONTROLLI DI QUALITÀ ---------------------------------------------------
# Se un controllo "bloccante" fallisce, il notebook si ferma e NON scrive tabelle.
CHIAVE = ["noga_cod", "paese_cod", "sesso_cod", "cantone_cod", "trimestre"]

def controlli(df, nome):
    print(f"== {nome} ==")
    dup = df.duplicated(CHIAVE).sum()
    print(f"[bloccante] righe duplicate sulla chiave: {dup}")
    assert dup == 0
    nulli = df["frontalieri"].isna().sum()
    print(f"[info] valori mancanti: {nulli} su {len(df)}")
    neg = (df["frontalieri"] < 0).sum()
    print(f"[bloccante] valori negativi: {neg}")
    assert neg == 0
    attese = 1
    for c in ["noga_cod", "paese_cod", "cantone_cod", "trimestre"]:
        attese *= df[c].nunique()
    print(f"[info] righe {len(df)} / combinazioni attese {attese} -> {'completo' if len(df)==attese else 'INCOMPLETO'}")
    print(f"[info] periodo: {df['trimestre'].min()} -> {df['trimestre'].max()} ({df['trimestre'].nunique()} trimestri)")

controlli(cantone, "cantone")
controlli(settori, "settori")

# Coerenza: la somma dei cantoni deve ricostruire il totale svizzero (tolleranza 1%)
tot = cantone[(cantone.paese_cod == "_T") & (cantone.cantone_cod == "_T")].set_index("trimestre")["frontalieri"]
somma = (cantone[(cantone.paese_cod == "_T") & (cantone.cantone_cod != "_T")]
         .groupby("trimestre")["frontalieri"].sum())
scarto = ((somma - tot).abs() / tot).max()
print(f"[bloccante] scarto massimo somma cantoni vs totale CH: {scarto:.4%}")
assert scarto < 0.01


In [ ]:
# --- 4. SCRITTURA TABELLE DELTA (Silver) ---------------------------------------
# overwrite = idempotente: rieseguire dà lo stesso risultato. La storia resta
# nei file Bronze (uno per download) e nella cronologia Delta (time travel).
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
for df, tabella in [(cantone, "silver_frontalieri_cantone"), (settori, "silver_frontalieri_settori")]:
    (spark.createDataFrame(df)
          .write.mode("overwrite").option("overwriteSchema", "true")
          .saveAsTable(f"{CATALOG}.{SCHEMA}.{tabella}"))
    n = spark.table(f"{CATALOG}.{SCHEMA}.{tabella}").count()
    print(f"OK {CATALOG}.{SCHEMA}.{tabella}: {n} righe")


In [ ]:
# --- 5. PRIMA OCCHIATA: frontalieri residenti in Italia, Ticino vs Svizzera ----
display(spark.sql(f"""
  SELECT trimestre,
         ROUND(SUM(CASE WHEN cantone_cod = '21' THEN frontalieri END)) AS ticino_da_italia,
         ROUND(SUM(CASE WHEN cantone_cod = '_T' THEN frontalieri END)) AS svizzera_da_italia
  FROM {CATALOG}.{SCHEMA}.silver_frontalieri_cantone
  WHERE paese_cod = 'IT' AND trimestre >= '2021-Q1'
  GROUP BY trimestre ORDER BY trimestre
"""))
